# V17 – Kryptografie: Praxis

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- mit `ord()` und `chr()` flüssig zwischen Zeichen und ASCII-Code wechseln,
- eine Hilfsfunktion `caesar_zeichen(z, k)` für einen einzelnen Großbuchstaben implementieren,
- die Funktion `caesar_text(text, k)` zeichenweise auf einen ganzen String anwenden,
- Caesar durch Umkehren des Schlüssels wieder entschlüsseln,
- einen einfachen **Brute-Force-Angriff** per Schleife programmieren.

## ⏱️ Zeitbudget
Ca. 20 Minuten.

## 🧭 TL;DR
Wir bauen Caesar Schritt für Schritt zusammen: zuerst `ord`/`chr`, dann die Positionsformel, dann die Zeichen- und Text-Funktion, zum Schluss ein kleiner Knack-Angriff.

## 📦 Voraussetzungen
- [00_python_recap.ipynb](00_python_recap.ipynb)
- [01_theorie.ipynb](01_theorie.ipynb)

## 1. `ord()` und `chr()` sicher einsetzen

`ord("A")` liefert `65`, `chr(65)` liefert `"A"`. Die beiden Funktionen sind zueinander invers, solange man im ASCII-Bereich bleibt. Für V17 interessiert uns nur der Bereich der **Großbuchstaben**, also 65 bis 90.

In [1]:
print(ord("A"), ord("Z"))        # 65 90
print(chr(65), chr(90))          # A Z
print(chr(ord("H")))              # H – rundtrip
print(ord(chr(75)))               # 75 – rundtrip

65 90
A Z
H
75


### Kleine ASCII-Tabelle

Lass dir einmal alle Großbuchstaben mit zugehörigem Code ausgeben. So siehst du, wie sauber der Bereich `A`–`Z` auf 65–90 abgebildet ist.

In [2]:
for code in range(65, 91):
    print(f"{code:3d} -> {chr(code)}")

 65 -> A
 66 -> B
 67 -> C
 68 -> D
 69 -> E
 70 -> F
 71 -> G
 72 -> H
 73 -> I
 74 -> J
 75 -> K
 76 -> L
 77 -> M
 78 -> N
 79 -> O
 80 -> P
 81 -> Q
 82 -> R
 83 -> S
 84 -> T
 85 -> U
 86 -> V
 87 -> W
 88 -> X
 89 -> Y
 90 -> Z


### ✏️ Fill-in 1 – ASCII-Code vom Buchstaben `M`

Setze die Variable `code_m` auf den ASCII-Code des Buchstabens `M`. Nutze `ord()`.

In [3]:
# TODO: code_m = ord("M")
code_m = 0
print("code_m =", code_m)

code_m = 0


In [4]:
try:
    assert code_m == 77, f"Erwartet 77, bekommen {code_m}"
    print("✅ ord('M') == 77.")
except AssertionError as e:
    print(f"❌ {e}")
except NameError:
    print("❌ Die Variable `code_m` existiert noch nicht.")

❌ Erwartet 77, bekommen 0


## 2. Vom Zeichen zur Alphabet-Position

Die Caesar-Formel arbeitet nicht mit ASCII-Codes, sondern mit **Positionen 0–25**. Zwei Schritte reichen:

1. **Zeichen → Position:** `pos = ord(z) - 65`
2. **Position → Zeichen:** `z = chr(pos + 65)`

Mit diesen beiden Umrechnungen hast du eine sichere Brücke zwischen Buchstaben und Zahlen.

In [5]:
for z in "AHMZ":
    pos = ord(z) - 65
    zurueck = chr(pos + 65)
    print(f"{z} -> Position {pos:2d} -> {zurueck}")

A -> Position  0 -> A
H -> Position  7 -> H
M -> Position 12 -> M
Z -> Position 25 -> Z


## 3. `caesar_zeichen(z, k)` – ein einzelner Buchstabe

Wir bauen jetzt eine kleine Funktion, die einen **einzelnen** Großbuchstaben `z` um `k` Schritte verschiebt. Nicht-Buchstaben (Leerzeichen, Ziffern, Punkte) lassen wir unverändert – das macht die spätere Text-Funktion viel angenehmer.

In [6]:
def caesar_zeichen(z, k):
    if "A" <= z <= "Z":
        pos = ord(z) - 65
        neu = (pos + k) % 26
        return chr(neu + 65)
    return z

print(caesar_zeichen("A", 3))   # D
print(caesar_zeichen("H", 3))   # K
print(caesar_zeichen("Z", 1))   # A  – Umlauf
print(caesar_zeichen(" ", 3))   # ' ' unverändert
print(caesar_zeichen("!", 5))   # '!' unverändert

D
K
A
 
!


### ✏️ Fill-in 2 – Eigene Variante mit Schlüssel 7

Rufe `caesar_zeichen` für den Buchstaben `"T"` mit Schlüssel `7` auf und speichere das Ergebnis in `zeichen_7`. Erwartet: `"A"` (Position 19 + 7 = 26, `% 26` = 0 → `A`).

In [7]:
# TODO: Rufe caesar_zeichen("T", 7) auf
zeichen_7 = ""
print("zeichen_7 =", zeichen_7)

zeichen_7 = 


In [8]:
try:
    assert zeichen_7 == "A", f"Erwartet 'A', bekommen '{zeichen_7}'"
    print("✅ T um 7 verschoben landet bei A.")
except AssertionError as e:
    print(f"❌ {e}")
except NameError:
    print("❌ Die Variable `zeichen_7` existiert noch nicht.")

❌ Erwartet 'A', bekommen ''


## 4. `caesar_text(text, k)` – ein ganzer String

Jetzt legen wir ein Schleifchen um `caesar_zeichen`, damit wir einen beliebigen Text auf einen Rutsch verschlüsseln können. Weil `caesar_zeichen` Nicht-Buchstaben unverändert lässt, funktioniert die Text-Funktion auch für Sätze mit Leerzeichen und Satzzeichen.

In [9]:
def caesar_text(text, k):
    ergebnis = ""
    for z in text:
        ergebnis += caesar_zeichen(z, k)
    return ergebnis

print(caesar_text("HALLO WELT", 3))                   # KDOOR ZHOW
print(caesar_text("VERSCHLUESSELUNG IST WICHTIG", 4))
print(caesar_text("ABCXYZ", 1))                        # BCDYZA

KDOOR ZHOW
ZIVWGLPYIWWIPYRK MWX AMGLXMK
BCDYZA


### ✏️ Fill-in 3 – Verschlüssele `"MASCHINE"` mit Schlüssel 5

Berechne `caesar_text("MASCHINE", 5)` und speichere das Ergebnis in `geheim_maschine`. Erwartet: `"RFXHMNSJ"`.

In [10]:
# TODO: caesar_text("MASCHINE", 5)
geheim_maschine = ""
print("geheim_maschine =", geheim_maschine)

geheim_maschine = 


In [11]:
try:
    assert geheim_maschine == "RFXHMNSJ", f"Erwartet 'RFXHMNSJ', bekommen '{geheim_maschine}'"
    print("✅ Verschlüsselung passt.")
except AssertionError as e:
    print(f"❌ {e}")
except NameError:
    print("❌ Die Variable `geheim_maschine` existiert noch nicht.")

❌ Erwartet 'RFXHMNSJ', bekommen ''


## 5. Entschlüsseln – einfach das Vorzeichen drehen

Weil Caesar eine reine Verschiebung ist, reicht es, mit **`-k`** erneut zu verschlüsseln, um den Klartext zu erhalten. Alternativ kannst du `(26 - k) % 26` als „Rück-Schlüssel“ verwenden – beide Wege führen zum selben Ergebnis.

In [12]:
geheim = caesar_text("ANGRIFF BEI MORGENGRAUEN", 7)
print("Geheim:", geheim)
print("Klar (-k):      ", caesar_text(geheim, -7))
print("Klar (26-k) % 26:", caesar_text(geheim, (26 - 7) % 26))

Geheim: HUNYPMM ILP TVYNLUNYHBLU
Klar (-k):       ANGRIFF BEI MORGENGRAUEN
Klar (26-k) % 26: ANGRIFF BEI MORGENGRAUEN


## 6. Mini-Challenge – Brute-Force eines fremden Geheimtexts

Jemand hat dir folgenden Geheimtext zugesteckt – du kennst den Schlüssel nicht:

```
KDOOR YRP JHKHLPHQ GLHQVW
```

Neee, so ergibt das nichts. Aber du weißt, dass es sich um Caesar handelt. Dafür gibt es **genau 25 mögliche Schlüssel**, und du probierst sie einfach alle aus. Der einzige lesbare Text unter den 25 Ergebnissen ist der Klartext.

Führe die nächste Zelle aus und suche in der Ausgabe die Zeile, die sich wie deutscher Text liest.

In [13]:
geheim = "KDOOR YRP JHKHLPHQ GLHQVW"
for k in range(1, 26):
    versuch = caesar_text(geheim, -k)
    print(f"k = {k:2d}: {versuch}")

k =  1: JCNNQ XQO IGJGKOGP FKGPUV
k =  2: IBMMP WPN HFIFJNFO EJFOTU
k =  3: HALLO VOM GEHEIMEN DIENST
k =  4: GZKKN UNL FDGDHLDM CHDMRS
k =  5: FYJJM TMK ECFCGKCL BGCLQR
k =  6: EXIIL SLJ DBEBFJBK AFBKPQ
k =  7: DWHHK RKI CADAEIAJ ZEAJOP
k =  8: CVGGJ QJH BZCZDHZI YDZINO
k =  9: BUFFI PIG AYBYCGYH XCYHMN
k = 10: ATEEH OHF ZXAXBFXG WBXGLM
k = 11: ZSDDG NGE YWZWAEWF VAWFKL
k = 12: YRCCF MFD XVYVZDVE UZVEJK
k = 13: XQBBE LEC WUXUYCUD TYUDIJ
k = 14: WPAAD KDB VTWTXBTC SXTCHI
k = 15: VOZZC JCA USVSWASB RWSBGH
k = 16: UNYYB IBZ TRURVZRA QVRAFG
k = 17: TMXXA HAY SQTQUYQZ PUQZEF
k = 18: SLWWZ GZX RPSPTXPY OTPYDE
k = 19: RKVVY FYW QOROSWOX NSOXCD
k = 20: QJUUX EXV PNQNRVNW MRNWBC
k = 21: PITTW DWU OMPMQUMV LQMVAB
k = 22: OHSSV CVT NLOLPTLU KPLUZA
k = 23: NGRRU BUS MKNKOSKT JOKTYZ
k = 24: MFQQT ATR LJMJNRJS INJSXY
k = 25: LEPPS ZSQ KILIMQIR HMIRWX


> 💡 **Tipp:** Die lesbare Zeile enthält die Worte `HALLO` und `DIENST` – prüfe den zugehörigen Schlüssel. Genau so händisch gehen Angreifer bei Caesar vor – 25 Sekunden reichen.

## ✅ Zusammenfassung

- `caesar_zeichen(z, k)` verschiebt einen Großbuchstaben um `k` Schritte mit `% 26`.
- `caesar_text(text, k)` wendet das zeichenweise auf einen String an.
- Entschlüsseln = `caesar_text(geheim, -k)`.
- Brute-Force = alle 25 Schlüssel ausprobieren und den lesbaren Text auswählen.

## ➡️ Nächster Schritt
Weiter mit [03_aufgaben.ipynb](03_aufgaben.ipynb) – fünf Übungsaufgaben zur Selbstkontrolle.